# DSS Final Model Export — Full Assessment-Guided LSTM

Notebook ini **bukan** notebook eksperimen penuh. Tujuannya hanya melatih **satu model final** untuk website DSS dari dataset fitur yang sudah dihitung pada eksperimen final.

Konfigurasi yang dipakai:
- Full Assessment-Guided LSTM
- 122 fitur
- window 24 timestep = 6 jam
- horizon 15/30/45/60 menit
- split utama 70/15/15 kronologis per kolam
- 96 LSTM units
- dropout 0.20
- seed 42
- target DO tetap dalam mg/L (tidak di-scale)

Output:
- `model_lstm_do.keras`
- `scaler_x.pkl`
- `model_metadata.json`
- `DSS_LSTM_ARTIFACTS.zip`


In [ ]:
# 1. Setup
import os, json, random, zipfile, platform
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)

SEED = 42
WINDOW = 24
HORIZONS = [1, 2, 3, 4]
HORIZON_MINUTES = [15, 30, 45, 60]
INTERVAL_MINUTES = 15
UNITS = 96
DROPOUT = 0.20
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
EPOCHS = 120
PATIENCE = 15

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

set_all_seeds(SEED)
print({'seed':SEED,'window':WINDOW,'units':UNITS,'dropout':DROPOUT,'horizons_min':HORIZON_MINUTES})


In [ ]:
# 2. Exact 122-feature order used by the DSS backend
FEATURES = [
  "temp",
  "ph",
  "do",
  "hour_sin",
  "hour_cos",
  "day_index",
  "pond_Catfish",
  "pond_Gourami",
  "pond_Nilem",
  "pond_Tilapia",
  "temp_lag1",
  "temp_lag2",
  "temp_lag3",
  "temp_lag4",
  "temp_lag8",
  "temp_lag12",
  "temp_lag24",
  "temp_roll_mean4",
  "temp_roll_min4",
  "temp_roll_max4",
  "temp_roll_mean8",
  "temp_roll_min8",
  "temp_roll_max8",
  "temp_roll_mean24",
  "temp_roll_min24",
  "temp_roll_max24",
  "ph_lag1",
  "ph_lag2",
  "ph_lag3",
  "ph_lag4",
  "ph_lag8",
  "ph_lag12",
  "ph_lag24",
  "ph_roll_mean4",
  "ph_roll_min4",
  "ph_roll_max4",
  "ph_roll_mean8",
  "ph_roll_min8",
  "ph_roll_max8",
  "ph_roll_mean24",
  "ph_roll_min24",
  "ph_roll_max24",
  "do_lag1",
  "do_lag2",
  "do_lag3",
  "do_lag4",
  "do_lag8",
  "do_lag12",
  "do_lag24",
  "do_roll_mean4",
  "do_roll_min4",
  "do_roll_max4",
  "do_roll_mean8",
  "do_roll_min8",
  "do_roll_max8",
  "do_roll_mean24",
  "do_roll_min24",
  "do_roll_max24",
  "temp_score",
  "ph_score",
  "do_score",
  "pond_score",
  "do_gap_ops",
  "do_gap_opt",
  "temp_score_roll_mean4",
  "temp_score_roll_sum4",
  "temp_score_roll_mean8",
  "temp_score_roll_sum8",
  "temp_score_roll_mean24",
  "temp_score_roll_sum24",
  "ph_score_roll_mean4",
  "ph_score_roll_sum4",
  "ph_score_roll_mean8",
  "ph_score_roll_sum8",
  "ph_score_roll_mean24",
  "ph_score_roll_sum24",
  "do_score_roll_mean4",
  "do_score_roll_sum4",
  "do_score_roll_mean8",
  "do_score_roll_sum8",
  "do_score_roll_mean24",
  "do_score_roll_sum24",
  "pond_score_roll_mean4",
  "pond_score_roll_sum4",
  "pond_score_roll_mean8",
  "pond_score_roll_sum8",
  "pond_score_roll_mean24",
  "pond_score_roll_sum24",
  "do_excursion_flag",
  "do_warning_flag",
  "do_excursion_duration_steps",
  "do_excursion_duration_minutes",
  "do_excursion_flag_roll_mean4",
  "do_excursion_flag_roll_sum4",
  "do_excursion_flag_roll_mean8",
  "do_excursion_flag_roll_sum8",
  "do_excursion_flag_roll_mean24",
  "do_excursion_flag_roll_sum24",
  "do_warning_flag_roll_mean4",
  "do_warning_flag_roll_sum4",
  "do_warning_flag_roll_mean8",
  "do_warning_flag_roll_sum8",
  "do_warning_flag_roll_mean24",
  "do_warning_flag_roll_sum24",
  "temp_roll_sd4",
  "temp_roll_cv4",
  "temp_roll_sd8",
  "temp_roll_cv8",
  "temp_roll_sd24",
  "temp_roll_cv24",
  "ph_roll_sd4",
  "ph_roll_cv4",
  "ph_roll_sd8",
  "ph_roll_cv8",
  "ph_roll_sd24",
  "ph_roll_cv24",
  "do_roll_sd4",
  "do_roll_cv4",
  "do_roll_sd8",
  "do_roll_cv8",
  "do_roll_sd24",
  "do_roll_cv24"
]
POND_COLUMNS = [
  "pond_Catfish",
  "pond_Gourami",
  "pond_Nilem",
  "pond_Tilapia"
]
FEATURE_GROUP_SIZES = {
  "BASE_FEATURES": 10,
  "TEMPORAL_FEATURES": 48,
  "SUITABILITY_FEATURES": 30,
  "EXCURSION_FEATURES": 16,
  "STABILITY_FEATURES": 18
}
assert len(FEATURES) == 122
print('Feature count:', len(FEATURES))


In [ ]:
# 3. Load precomputed feature dataset
CSV_PATH = 'processed_iot_dataset_with_features.csv'
if not os.path.exists(CSV_PATH):
    try:
        from google.colab import files
        print('Upload: processed_iot_dataset_with_features.csv')
        uploaded = files.upload()
        if CSV_PATH not in uploaded:
            raise FileNotFoundError(CSV_PATH)
    except Exception as e:
        raise FileNotFoundError('Upload processed_iot_dataset_with_features.csv ke runtime Colab.') from e

df = pd.read_csv(CSV_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
missing = [c for c in FEATURES if c not in df.columns]
assert not missing, f'Missing features: {missing}'
assert set(df['pond'].dropna().unique()) == {'Catfish','Gourami','Nilem','Tilapia'}
print('Dataset shape:', df.shape)
print('All 122 required features found.')


In [ ]:
# 4. Build sequence exactly as in Revised_Design_Experiment.ipynb
def make_sequences(data, feature_cols, window=WINDOW, horizons=HORIZONS):
    Xs, Ys, metas = [], [], []
    max_h = max(horizons)
    for pond, g in data.groupby('pond'):
        g = g.sort_values('timestamp').reset_index(drop=True)
        feature_values = g[feature_cols].values.astype(float)
        do_values = g['do'].values.astype(float)
        timestamps = g['timestamp'].values
        for end in range(window - 1, len(g) - max_h):
            start = end - window + 1
            X = feature_values[start:end+1]
            y = np.array([do_values[end+h] for h in horizons], dtype=float)
            if np.isnan(X).any() or np.isnan(y).any():
                continue
            Xs.append(X); Ys.append(y)
            metas.append({'pond':pond,'end_timestamp':pd.Timestamp(timestamps[end])})
    return np.asarray(Xs,float), np.asarray(Ys,float), pd.DataFrame(metas)

def chronological_split(meta, train_ratio=0.70, val_ratio=0.15):
    tr, va, te = [], [], []
    for pond, g in meta.groupby('pond'):
        idx = g.sort_values('end_timestamp').index.to_numpy()
        n = len(idx); ntr = int(np.floor(train_ratio*n)); nva = int(np.floor(val_ratio*n))
        tr.extend(idx[:ntr]); va.extend(idx[ntr:ntr+nva]); te.extend(idx[ntr+nva:])
    return np.asarray(tr), np.asarray(va), np.asarray(te)

X, Y, meta_seq = make_sequences(df, FEATURES)
train_idx, val_idx, test_idx = chronological_split(meta_seq)
print('X:', X.shape, 'Y:', Y.shape)
print('Train:',len(train_idx),'Val:',len(val_idx),'Test:',len(test_idx))
assert X.shape[1:] == (24,122)
assert Y.shape[1] == 4


In [ ]:
# 5. Fit scaler on TRAIN X only
X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
y_train, y_val, y_test = Y[train_idx], Y[val_idx], Y[test_idx]

scaler_x = StandardScaler()
_, t, f = X_train.shape
scaler_x.fit(X_train.reshape(-1,f))

def transform_3d(A):
    n,t,f = A.shape
    return scaler_x.transform(A.reshape(-1,f)).reshape(n,t,f)

X_train_s = transform_3d(X_train)
X_val_s = transform_3d(X_val)
X_test_s = transform_3d(X_test)
print('Scaled:',X_train_s.shape,X_val_s.shape,X_test_s.shape)
print('Y remains raw DO mg/L.')


In [ ]:
# 6. Train ONE final Full Assessment-Guided LSTM
def build_lstm(input_shape, output_dim=4, units=96, dropout=0.20, lr=1e-3):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(units, return_sequences=False),
        layers.Dropout(dropout),
        layers.Dense(32, activation='relu'),
        layers.Dense(output_dim)
    ])
    model.compile(optimizer=optimizers.Adam(learning_rate=lr), loss='mse', metrics=['mae'])
    return model

set_all_seeds(SEED)
model = build_lstm(X_train_s.shape[1:], units=UNITS, dropout=DROPOUT, lr=LEARNING_RATE)
cb = [
    callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', patience=max(2,PATIENCE//2), factor=0.5, min_lr=1e-5)
]
history = model.fit(X_train_s,y_train,validation_data=(X_val_s,y_val),epochs=EPOCHS,batch_size=BATCH_SIZE,verbose=1,callbacks=cb)
print('Epochs run:',len(history.history['loss']))
print('Best val MSE:',min(history.history['val_loss']))


In [ ]:
# 7. Sanity evaluation on held-out test split
y_pred = model.predict(X_test_s, verbose=0)
rows=[]
for j,minute in enumerate(HORIZON_MINUTES):
    yt=y_test[:,j]; yp=y_pred[:,j]
    rows.append({
        'horizon_min':minute,
        'MAE':mean_absolute_error(yt,yp),
        'RMSE':np.sqrt(mean_squared_error(yt,yp)),
        'MAPE_pct':np.mean(np.abs((yt-yp)/np.where(yt==0,np.nan,yt)))*100,
        'R2':r2_score(yt,yp)
    })
metrics_df=pd.DataFrame(rows)
display(metrics_df)
assert y_pred.shape[1]==4 and np.isfinite(y_pred).all()
print('Prediction range mg/L:',float(y_pred.min()),'to',float(y_pred.max()))


In [ ]:
# 8. Export artifacts
MODEL_PATH='model_lstm_do.keras'
SCALER_PATH='scaler_x.pkl'
META_PATH='model_metadata.json'
ZIP_PATH='DSS_LSTM_ARTIFACTS.zip'

model.save(MODEL_PATH)
joblib.dump(scaler_x, SCALER_PATH)

model_metadata={
    'model_variant':'assessment_guided_lstm_multi_horizon',
    'source':'DSS_Final_Model_Export.ipynb derived from Revised_Design_Experiment.ipynb',
    'training_scenario':'Full_Assessment_Guided_LSTM',
    'export_fold':0,
    'export_seed':SEED,
    'window_size':WINDOW,
    'window_minutes':WINDOW*INTERVAL_MINUTES,
    'sampling_interval_minutes':INTERVAL_MINUTES,
    'horizons':HORIZON_MINUTES,
    'target_scaling':'none',
    'target_scaling_note':'Y trained directly in DO mg/L; only X is standardized.',
    'architecture':{
        'lstm_units':UNITS,'dropout':DROPOUT,'dense_units':32,'output_units':4,
        'optimizer':'Adam','learning_rate_initial':LEARNING_RATE,'loss':'mse',
        'batch_size':BATCH_SIZE,'early_stopping_patience':PATIENCE
    },
    'features':FEATURES,
    'pond_columns':POND_COLUMNS,
    'feature_group_sizes':FEATURE_GROUP_SIZES,
    'test_metrics':metrics_df.to_dict(orient='records')
}
with open(META_PATH,'w') as f: json.dump(model_metadata,f,indent=2)

with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(MODEL_PATH); z.write(SCALER_PATH); z.write(META_PATH)

reloaded=tf.keras.models.load_model(MODEL_PATH)
assert tuple(reloaded.input_shape[1:])==(24,122)
assert reloaded.output_shape[-1]==4
print('Reload check passed:',reloaded.input_shape,'->',reloaded.output_shape)
print('Created:',MODEL_PATH,SCALER_PATH,META_PATH,ZIP_PATH)


In [ ]:
# 9. Auto-download final ZIP in Colab
try:
    from google.colab import files
    files.download('DSS_LSTM_ARTIFACTS.zip')
except Exception:
    print('Download DSS_LSTM_ARTIFACTS.zip manually from the Colab file browser.')
